# DenseNet169 — ChestX-ray14 Multi-Label Classification

**Dataset:** NIH ChestX-ray14 — 112,120 frontal-view chest X-rays, 14 pathology labels  
**Model:** DenseNet169 (ImageNet pre-trained)  
**Split:** Patient-level stratified 70 / 10 / 20 % (Wang et al. 2017 strategy)  
**Hardware target:** Single GPU ≤ 8 GB VRAM  

---
### Expected directory structure
```
DATASET_ROOT/
├── Data_Entry_2017_v2020.csv
├── images_001/
│   └── images/
│         └── *.png
├── images_002/
│   └── images/
│         └── *.png
│   ...
└── images_012/
    └── images/
          └── *.png
```
No `train_val_list.txt` / `test_list.txt` required — split is derived from the CSV using patient IDs.

## 0. Imports & Reproducibility

In [ ]:
import os
import random
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

import torchvision.transforms as T
import torchvision.models as models

from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score,
    recall_score, precision_score,
    roc_curve, precision_recall_curve,
    multilabel_confusion_matrix,
)
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

# ── Device ─────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU  : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Configuration

In [ ]:
# ─── CHANGE THIS to your dataset root folder ───────────────────────────────────
DATASET_ROOT = Path(r'D:\ChestX-ray14 Complete')   # Windows example
# DATASET_ROOT = Path('/data/ChestX-ray14')         # Linux/macOS example
# ──────────────────────────────────────────────────────────────────────────────

CSV_PATH = DATASET_ROOT / 'Data_Entry_2017_v2020.csv'
CKPT_DIR = Path('./checkpoints')
CKPT_DIR.mkdir(exist_ok=True)

# ── 14 pathology labels ───────────────────────────────────────────────────────
LABELS: List[str] = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
    'Effusion',    'Emphysema',    'Fibrosis',       'Hernia',
    'Infiltration','Mass',         'Nodule',         'Pleural_Thickening',
    'Pneumonia',   'Pneumothorax',
]
NUM_CLASSES = len(LABELS)

# ── Patient-level split (Wang et al. 2017 proportions) ───────────────────────
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.10
# TEST_RATIO  = 0.20  (implicit)

# ── Training ──────────────────────────────────────────────────────────────────
IMG_SIZE     = 224
BATCH_SIZE   = 32      # reduce to 16 if OOM on 8 GB VRAM
NUM_WORKERS  = 4
PIN_MEMORY   = True

LR_BACKBONE  = 1e-4   # pre-trained backbone
LR_HEAD      = 1e-3   # new classifier head
WEIGHT_DECAY = 1e-5
NUM_EPOCHS   = 30
PATIENCE     = 7      # early-stopping
USE_AMP      = True   # mixed precision — critical for 8 GB VRAM
THRESHOLD    = 0.5    # sigmoid threshold for binary decisions

assert CSV_PATH.exists(), f'CSV not found: {CSV_PATH}'
print('Configuration ready.')
print(f'  Root   : {DATASET_ROOT}')
print(f'  CSV    : {CSV_PATH.name}')
print(f'  Split  : {TRAIN_RATIO:.0%} train / {VAL_RATIO:.0%} val / {1-TRAIN_RATIO-VAL_RATIO:.0%} test')

## 2. Image Discovery — Recursive Subdirectory Walk

In [ ]:
def build_image_index(root: Path) -> Dict[str, Path]:
    """
    Recursively indexes every .png file under `root`.
    Handles the NIH layout: images_001/images/*.png ... images_012/images/*.png
    Returns {filename: full_path}.
    """
    index: Dict[str, Path] = {}
    for p in root.rglob('*.png'):
        index[p.name] = p
    if not index:
        raise FileNotFoundError(
            f'No .png files found under {root}. '
            'Double-check DATASET_ROOT.'
        )
    print(f'Indexed {len(index):,} PNG images')
    return index


image_index = build_image_index(DATASET_ROOT)

## 3. CSV Inspection & Normalisation

In [ ]:
raw_df = pd.read_csv(CSV_PATH)
print(f'CSV rows x cols : {raw_df.shape}')
print(f'Columns         : {list(raw_df.columns)}')
raw_df.head(3)

In [ ]:
# ── Normalise column names (handles minor version differences in the NIH CSV) ─
# Data_Entry_2017_v2020.csv has:
#   'Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID',
#   'Patient Age', 'Patient Gender', 'View Position',
#   'OriginalImageWidth', 'OriginalImageHeight',
#   'OriginalImagePixelSpacing_x', 'OriginalImagePixelSpacing_y'
RENAME = {
    'Image Index'    : 'image_index',
    'Finding Labels' : 'finding_labels',
    'Patient ID'     : 'patient_id',
    'Patient Age'    : 'patient_age',
    'Patient Gender' : 'patient_gender',
    'View Position'  : 'view_position',
}
raw_df.rename(columns=RENAME, inplace=True)

# ── Keep only frontal views (PA / AP) ─────────────────────────────────────────
if 'view_position' in raw_df.columns:
    before = len(raw_df)
    raw_df = raw_df[raw_df['view_position'].isin(['PA', 'AP'])].reset_index(drop=True)
    print(f'Frontal-view filter: kept {len(raw_df):,} / {before:,} rows')

# ── Drop rows whose image file is absent from disk ────────────────────────────
before = len(raw_df)
raw_df = raw_df[raw_df['image_index'].isin(image_index)].reset_index(drop=True)
missing = before - len(raw_df)
if missing:
    print(f'WARNING: {missing} rows had no matching image on disk and were dropped.')

print(f'\nFinal dataframe  : {len(raw_df):,} rows')
print(f'Unique patients  : {raw_df["patient_id"].nunique():,}')
print(f'\nTop finding_labels:')
print(raw_df['finding_labels'].value_counts().head(8).to_string())

## 4. One-Hot Encoding & Patient-Level Split

In [ ]:
def one_hot_encode(df: pd.DataFrame) -> pd.DataFrame:
    """
    Vectorised one-hot encoding.
    Uses str.contains (no Python-level loop) — fast on 112k rows.
    'No Finding' rows produce all-zero label vectors, which is correct.
    """
    df = df.copy()
    for label in LABELS:
        df[label] = df['finding_labels'].str.contains(label, regex=False).astype(np.float32)
    return df


def patient_level_split(
    df:          pd.DataFrame,
    train_ratio: float = TRAIN_RATIO,
    val_ratio:   float = VAL_RATIO,
    seed:        int   = SEED,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Guarantees zero patient overlap across splits — the primary data-leakage
    risk for ChestX-ray14, where one patient can have 1–dozens of images.

    Replicates the Wang et al. (2017) 70/10/20 strategy.
    GroupShuffleSplit ensures reproducibility via `seed`.
    """
    patients = df['patient_id'].values
    idx      = np.arange(len(df))

    # Step 1 — carve out 20 % test
    gss1 = GroupShuffleSplit(n_splits=1, test_size=1.0 - train_ratio - val_ratio, random_state=seed)
    trainval_idx, test_idx = next(gss1.split(idx, groups=patients))

    # Step 2 — split remaining 80 % into 70 % train / 10 % val
    val_frac = val_ratio / (train_ratio + val_ratio)   # 10/80 = 0.125
    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_frac, random_state=seed)
    train_sub, val_sub = next(gss2.split(trainval_idx, groups=patients[trainval_idx]))
    train_idx = trainval_idx[train_sub]
    val_idx   = trainval_idx[val_sub]

    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df   = df.iloc[val_idx  ].reset_index(drop=True)
    test_df  = df.iloc[test_idx ].reset_index(drop=True)

    # ── Leakage assertions ────────────────────────────────────────────────────
    p_train, p_val, p_test = (
        set(train_df['patient_id']),
        set(val_df  ['patient_id']),
        set(test_df ['patient_id']),
    )
    assert not (p_train & p_test), 'LEAKAGE: train ∩ test'
    assert not (p_val   & p_test), 'LEAKAGE: val ∩ test'
    assert not (p_train & p_val),  'LEAKAGE: train ∩ val'
    print('Patient-level split — zero leakage confirmed.')

    for name, d in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
        print(f'  {name:5s}: {len(d):>7,} images | {d["patient_id"].nunique():>6,} patients')

    return train_df, val_df, test_df


df = one_hot_encode(raw_df)
train_df, val_df, test_df = patient_level_split(df)

### 4.1 Label Distribution Across Splits

In [ ]:
def plot_label_distribution(splits: Dict[str, pd.DataFrame]) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(22, 4), sharey=False)
    for ax, (split_name, df_s) in zip(axes, splits.items()):
        counts = df_s[LABELS].sum().sort_values(ascending=False)
        sns.barplot(x=counts.index, y=counts.values, palette='Blues_r', ax=ax)
        ax.set_title(f'{split_name}  ({len(df_s):,} images)', fontsize=11)
        ax.set_ylabel('Positive samples')
        ax.tick_params(axis='x', rotation=55)
        for bar, v in zip(ax.patches, counts.values):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + counts.max() * 0.012,
                f'{int(v):,}', ha='center', va='bottom', fontsize=6.5
            )
    plt.suptitle('Positive-Label Counts per Split', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig('label_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_label_distribution({'Train': train_df, 'Val': val_df, 'Test': test_df})

print('\nPositive rates (%) per split:')
display(pd.DataFrame({
    'Train %': (train_df[LABELS].mean() * 100).round(2),
    'Val %'  : (val_df  [LABELS].mean() * 100).round(2),
    'Test %' : (test_df [LABELS].mean() * 100).round(2),
}))

## 5. Dataset & DataLoaders

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


def get_transforms(split: str) -> T.Compose:
    """Augmentations only for training. Val/Test are deterministic."""
    if split == 'train':
        return T.Compose([
            T.Resize(256),
            T.RandomCrop(IMG_SIZE),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomRotation(degrees=10),
            T.ColorJitter(brightness=0.15, contrast=0.15),
            T.ToTensor(),
            T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


class ChestXray14Dataset(Dataset):
    """
    Memory-efficient: images are loaded on demand from disk.
    Only the label matrix (N x 14 float32) is held in RAM — negligible.
    """

    def __init__(
        self,
        df:          pd.DataFrame,
        image_index: Dict[str, Path],
        transform:   Optional[T.Compose] = None,
    ) -> None:
        self.filenames   = df['image_index'].tolist()
        self.labels      = df[LABELS].values.astype(np.float32)  # (N, 14)
        self.image_index = image_index
        self.transform   = transform

    def __len__(self) -> int:
        return len(self.filenames)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        img_path = self.image_index[self.filenames[idx]]
        img      = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = torch.from_numpy(self.labels[idx])  # (14,) float32
        return img, label


def build_loaders(
    train_df:    pd.DataFrame,
    val_df:      pd.DataFrame,
    test_df:     pd.DataFrame,
    image_index: Dict[str, Path],
) -> Dict[str, DataLoader]:
    splits = {'train': train_df, 'val': val_df, 'test': test_df}
    loaders: Dict[str, DataLoader] = {}
    for name, df_s in splits.items():
        ds = ChestXray14Dataset(df_s, image_index, get_transforms(name))
        bs = BATCH_SIZE if name == 'train' else BATCH_SIZE * 2
        loaders[name] = DataLoader(
            ds,
            batch_size         = bs,
            shuffle            = (name == 'train'),
            num_workers        = NUM_WORKERS,
            pin_memory         = PIN_MEMORY and DEVICE.type == 'cuda',
            drop_last          = (name == 'train'),
            persistent_workers = NUM_WORKERS > 0,
        )
        print(f'{name:5s} — {len(ds):>7,} samples | {len(loaders[name]):>5,} batches | bs={bs}')
    return loaders


loaders = build_loaders(train_df, val_df, test_df, image_index)

## 6. Model — DenseNet169

In [ ]:
def build_densenet169(num_classes: int = NUM_CLASSES) -> nn.Module:
    """
    DenseNet169 with ImageNet1K weights.
    Replaces the 1000-class head with a Linear(1664 → num_classes) layer.

    NOTE: No sigmoid in the model graph.
    BCEWithLogitsLoss = sigmoid + BCE in one numerically stable CUDA kernel.
    Sigmoid is applied only at inference time (.sigmoid() on logits).
    """
    weights = models.DenseNet169_Weights.IMAGENET1K_V1
    model   = models.densenet169(weights=weights)
    in_feat = model.classifier.in_features   # 1664
    model.classifier = nn.Linear(in_feat, num_classes)
    return model


model     = build_densenet169().to(DEVICE)
total_p   = sum(p.numel() for p in model.parameters())
train_p   = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'DenseNet169 ready')
print(f'  Total params     : {total_p:,}')
print(f'  Trainable params : {train_p:,}')
print(f'  Classifier       : {model.classifier}')

## 7. Loss, Optimiser & Scheduler

In [ ]:
def compute_pos_weights(df: pd.DataFrame) -> torch.Tensor:
    """
    pos_weight_i = n_negative_i / n_positive_i

    Passed to BCEWithLogitsLoss so the loss function internally up-weights
    the positive class — no resampling needed, avoiding overfitting on
    rare classes (e.g. Hernia ~0.2 %).
    """
    pos = df[LABELS].sum().values.astype(np.float64)
    neg = len(df) - pos
    w   = neg / np.clip(pos, 1, None)
    return torch.tensor(w, dtype=torch.float32).to(DEVICE)


pos_weights = compute_pos_weights(train_df)
criterion   = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

# Differential LRs: 10x lower for pre-trained backbone
backbone_params = [p for n, p in model.named_parameters() if 'classifier' not in n]
head_params     = list(model.classifier.parameters())

optimizer = optim.AdamW(
    [{'params': backbone_params, 'lr': LR_BACKBONE},
     {'params': head_params,     'lr': LR_HEAD}],
    weight_decay=WEIGHT_DECAY,
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
scaler    = GradScaler(enabled=USE_AMP)

print('pos_weights per class (higher = rarer):')
for lbl, w in zip(LABELS, pos_weights.cpu().numpy()):
    print(f'  {lbl:<22s}: {w:6.2f}')

## 8. Training Loop

In [ ]:
def train_one_epoch(
    model:     nn.Module,
    loader:    DataLoader,
    optimizer: optim.Optimizer,
    criterion: nn.Module,
    scaler:    GradScaler,
) -> float:
    model.train()
    total_loss = 0.0
    for imgs, labels in tqdm(loader, desc='Train', leave=False):
        imgs, labels = (
            imgs.to(DEVICE, non_blocking=True),
            labels.to(DEVICE, non_blocking=True),
        )
        optimizer.zero_grad(set_to_none=True)   # faster than zero_grad()
        with autocast(enabled=USE_AMP):
            loss = criterion(model(imgs), labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(
    model:     nn.Module,
    loader:    DataLoader,
    criterion: nn.Module,
) -> Tuple[float, float]:
    """
    Returns (mean_loss, mean_auc).
    Probs accumulated on CPU — prevents VRAM build-up over large eval sets.
    """
    model.eval()
    total_loss     = 0.0
    all_probs: List[torch.Tensor]  = []
    all_labels: List[torch.Tensor] = []
    for imgs, labels in tqdm(loader, desc='Eval ', leave=False):
        imgs, labels = (
            imgs.to(DEVICE, non_blocking=True),
            labels.to(DEVICE, non_blocking=True),
        )
        with autocast(enabled=USE_AMP):
            logits = model(imgs)
            loss   = criterion(logits, labels)
        total_loss += loss.item()
        all_probs.append(logits.sigmoid().cpu())
        all_labels.append(labels.cpu())

    probs  = torch.cat(all_probs).numpy()   # (N, 14)
    truths = torch.cat(all_labels).numpy()  # (N, 14)

    aucs = [
        roc_auc_score(truths[:, i], probs[:, i])
        for i in range(NUM_CLASSES) if truths[:, i].sum() > 0
    ]
    return total_loss / len(loader), float(np.mean(aucs))


def fit(
    model:      nn.Module,
    loaders:    Dict[str, DataLoader],
    optimizer:  optim.Optimizer,
    scheduler,
    criterion:  nn.Module,
    scaler:     GradScaler,
    num_epochs: int  = NUM_EPOCHS,
    patience:   int  = PATIENCE,
    ckpt_path:  Path = CKPT_DIR / 'best_model.pth',
) -> Dict[str, List[float]]:

    history: Dict[str, List[float]] = {'train_loss': [], 'val_loss': [], 'val_auc': []}
    best_auc, stale = -1.0, 0

    for epoch in range(1, num_epochs + 1):
        tr_loss           = train_one_epoch(model, loaders['train'], optimizer, criterion, scaler)
        val_loss, val_auc = evaluate(model, loaders['val'], criterion)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(val_loss)
        history['val_auc'].append(val_auc)

        improved = val_auc > best_auc
        tag      = ' ← best' if improved else ''
        print(f'[{epoch:03d}/{num_epochs}]  '
              f'train_loss={tr_loss:.4f}  '
              f'val_loss={val_loss:.4f}  '
              f'val_auc={val_auc:.4f}{tag}')

        if improved:
            best_auc = val_auc
            stale    = 0
            torch.save({
                'epoch': epoch,
                'state_dict': model.state_dict(),
                'optimizer' : optimizer.state_dict(),
                'val_auc'   : best_auc,
            }, ckpt_path)
        else:
            stale += 1
            if stale >= patience:
                print(f'Early stopping triggered at epoch {epoch}.')
                break

    print(f'\nBest val AUC: {best_auc:.4f}  (ckpt → {ckpt_path})')
    return history


history = fit(model, loaders, optimizer, scheduler, criterion, scaler)

## 9. Learning Curves

In [ ]:
def plot_learning_curves(history: Dict[str, List[float]]) -> None:
    epochs = range(1, len(history['train_loss']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, history['train_loss'], marker='o', ms=3, label='Train')
    ax1.plot(epochs, history['val_loss'],   marker='o', ms=3, label='Val')
    ax1.set(title='Loss', xlabel='Epoch', ylabel='BCE Loss')
    ax1.legend(); ax1.grid(alpha=0.3)

    best = max(history['val_auc'])
    ax2.plot(epochs, history['val_auc'], marker='o', ms=3, color='darkorange', label='Val AUC')
    ax2.axhline(best, ls='--', color='red', alpha=0.7, label=f'Best = {best:.4f}')
    ax2.set(title='Validation Mean AUC', xlabel='Epoch', ylabel='AUC-ROC')
    ax2.legend(); ax2.grid(alpha=0.3)

    plt.suptitle('DenseNet169 — ChestX-ray14 Training', fontsize=13)
    plt.tight_layout()
    plt.savefig('learning_curves.png', dpi=150)
    plt.show()

plot_learning_curves(history)

## 10. Load Best Checkpoint & Run Test-Set Inference

In [ ]:
ckpt = torch.load(CKPT_DIR / 'best_model.pth', map_location=DEVICE)
model.load_state_dict(ckpt['state_dict'])
print(f'Loaded epoch {ckpt["epoch"]} — val AUC {ckpt["val_auc"]:.4f}')


@torch.no_grad()
def predict(
    model:  nn.Module,
    loader: DataLoader,
) -> Tuple[np.ndarray, np.ndarray]:
    """Returns (y_prob, y_true) — both shape (N, 14), accumulated on CPU."""
    model.eval()
    all_probs: List[torch.Tensor]  = []
    all_labels: List[torch.Tensor] = []
    for imgs, labels in tqdm(loader, desc='Test inference'):
        imgs = imgs.to(DEVICE, non_blocking=True)
        with autocast(enabled=USE_AMP):
            logits = model(imgs)
        all_probs.append(logits.sigmoid().cpu())
        all_labels.append(labels)
    return (
        torch.cat(all_probs).numpy(),
        torch.cat(all_labels).numpy(),
    )


y_prob, y_true = predict(model, loaders['test'])
y_pred = (y_prob >= THRESHOLD).astype(int)
print(f'Test predictions shape: {y_prob.shape}')

## 11. Per-Label Metrics Table

In [ ]:
def compute_metrics(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    y_pred: np.ndarray,
) -> pd.DataFrame:
    rows = []
    for i, lbl in enumerate(LABELS):
        gt, pb, pd_ = y_true[:, i], y_prob[:, i], y_pred[:, i]
        rows.append({
            'Label'      : lbl,
            'N_pos'      : int(gt.sum()),
            'Prevalence' : f'{gt.mean()*100:.2f}%',
            'AUC'        : round(roc_auc_score(gt, pb) if gt.sum() > 0 else float('nan'), 4),
            'F1'         : round(f1_score(gt, pd_, zero_division=0),        4),
            'Accuracy'   : round(accuracy_score(gt, pd_),                   4),
            'Recall'     : round(recall_score(gt, pd_, zero_division=0),    4),
            'Precision'  : round(precision_score(gt, pd_, zero_division=0), 4),
        })
    mdf = pd.DataFrame(rows)

    macro = pd.DataFrame([{
        'Label'      : 'MACRO AVG',
        'N_pos'      : int(y_true.sum()),
        'Prevalence' : f'{y_true.mean()*100:.2f}%',
        'AUC'        : round(mdf['AUC'].dropna().mean(), 4),
        'F1'         : round(mdf['F1'].mean(), 4),
        'Accuracy'   : round(mdf['Accuracy'].mean(), 4),
        'Recall'     : round(mdf['Recall'].mean(), 4),
        'Precision'  : round(mdf['Precision'].mean(), 4),
    }])
    return pd.concat([mdf, macro], ignore_index=True)


metrics_df = compute_metrics(y_true, y_prob, y_pred)

num_cols = ['AUC', 'F1', 'Accuracy', 'Recall', 'Precision']
display(
    metrics_df.style
        .format({c: '{:.4f}' for c in num_cols})
        .highlight_max(subset=num_cols, color='#c6efce')
        .highlight_min(subset=num_cols, color='#ffeb9c')
        .set_caption('Per-Label Test Metrics — DenseNet169 / ChestX-ray14')
)
metrics_df.to_csv('test_metrics.csv', index=False)
print('Saved → test_metrics.csv')

## 12. ROC Curves

In [ ]:
def plot_roc_curves(y_true: np.ndarray, y_prob: np.ndarray) -> None:
    n_cols = 4
    n_rows = (NUM_CLASSES + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4.2))
    axes = axes.flatten()

    for i, lbl in enumerate(LABELS):
        fpr, tpr, _ = roc_curve(y_true[:, i], y_prob[:, i])
        auc_val     = roc_auc_score(y_true[:, i], y_prob[:, i])
        ax = axes[i]
        ax.plot(fpr, tpr, lw=2, color='steelblue', label=f'AUC = {auc_val:.3f}')
        ax.fill_between(fpr, tpr, alpha=0.08, color='steelblue')
        ax.plot([0, 1], [0, 1], 'k--', lw=1)
        ax.set(title=lbl, xlabel='FPR', ylabel='TPR', xlim=[0, 1], ylim=[0, 1.02])
        ax.legend(loc='lower right', fontsize=9)
        ax.grid(alpha=0.3)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('ROC Curves — Per Pathology (Test Set)', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_roc_curves(y_true, y_prob)

## 13. Precision-Recall Curves

In [ ]:
def plot_pr_curves(y_true: np.ndarray, y_prob: np.ndarray) -> None:
    n_cols = 4
    n_rows = (NUM_CLASSES + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4.2))
    axes = axes.flatten()

    f1_lookup = dict(zip(
        metrics_df.loc[metrics_df['Label'] != 'MACRO AVG', 'Label'],
        metrics_df.loc[metrics_df['Label'] != 'MACRO AVG', 'F1'],
    ))

    for i, lbl in enumerate(LABELS):
        prec, rec, _ = precision_recall_curve(y_true[:, i], y_prob[:, i])
        baseline     = y_true[:, i].mean()
        ax = axes[i]
        ax.plot(rec, prec, lw=2, color='darkorange', label=f'F1={f1_lookup[lbl]:.3f}')
        ax.fill_between(rec, prec, alpha=0.08, color='darkorange')
        ax.axhline(baseline, ls='--', color='grey', lw=1, label=f'Prevalence={baseline:.3f}')
        ax.set(title=lbl, xlabel='Recall', ylabel='Precision', xlim=[0, 1], ylim=[0, 1.05])
        ax.legend(loc='upper right', fontsize=9)
        ax.grid(alpha=0.3)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Precision-Recall Curves — Per Pathology (Test Set)', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig('pr_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_pr_curves(y_true, y_prob)

## 14. Confusion Matrices

In [ ]:
def plot_confusion_matrices(y_true: np.ndarray, y_pred: np.ndarray) -> None:
    mcm    = multilabel_confusion_matrix(y_true, y_pred)   # (14, 2, 2)
    n_cols = 4
    n_rows = (NUM_CLASSES + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
    axes = axes.flatten()

    for i, lbl in enumerate(LABELS):
        cm      = mcm[i]                   # [[TN, FP], [FN, TP]]
        # Row-normalise so colour encodes rate, but raw counts are shown
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(1)
        ax = axes[i]
        sns.heatmap(
            cm_norm, annot=cm, fmt='d', cmap='Blues',
            vmin=0, vmax=1, cbar=False, ax=ax,
            xticklabels=['Pred Neg', 'Pred Pos'],
            yticklabels=['True Neg', 'True Pos'],
            annot_kws={'size': 11},
        )
        ax.set_title(lbl, fontsize=11)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(
        f'Confusion Matrices  (threshold={THRESHOLD}) — Test Set',
        fontsize=14, y=1.01
    )
    plt.tight_layout()
    plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_confusion_matrices(y_true, y_pred)

## 15. AUC Summary Bar Chart

In [ ]:
def plot_auc_bar(metrics_df: pd.DataFrame) -> None:
    df_p = (
        metrics_df[metrics_df['Label'] != 'MACRO AVG']
        .sort_values('AUC', ascending=False)
        .copy()
    )
    macro_auc = float(metrics_df.loc[metrics_df['Label'] == 'MACRO AVG', 'AUC'].iloc[0])

    fig, ax = plt.subplots(figsize=(14, 5))
    colors = plt.cm.RdYlGn(df_p['AUC'].values)
    bars   = ax.bar(df_p['Label'], df_p['AUC'], color=colors, edgecolor='black', lw=0.6)

    ax.axhline(macro_auc, ls='--', lw=2, color='navy',  label=f'Macro AUC = {macro_auc:.4f}')
    ax.axhline(0.5,       ls=':',  lw=1, color='red',   label='Random baseline = 0.50')

    for bar, val in zip(bars, df_p['AUC']):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.003,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8
        )

    ax.set(ylim=[0.4, 1.05], ylabel='AUC-ROC',
           title='Per-Pathology AUC-ROC — DenseNet169 / ChestX-ray14 Test Set')
    ax.tick_params(axis='x', rotation=45)
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('auc_bar.png', dpi=150)
    plt.show()

plot_auc_bar(metrics_df)

## 16. Final Summary

In [ ]:
macro = metrics_df[metrics_df['Label'] == 'MACRO AVG'].iloc[0]
print('=' * 60)
print('  DenseNet169 — ChestX-ray14 — Test Set Results')
print('=' * 60)
print(f'  Macro AUC       : {macro["AUC"]:.4f}')
print(f'  Macro F1        : {macro["F1"]:.4f}')
print(f'  Macro Accuracy  : {macro["Accuracy"]:.4f}')
print(f'  Macro Recall    : {macro["Recall"]:.4f}')
print(f'  Macro Precision : {macro["Precision"]:.4f}')
print('=' * 60)
print('\nPer-label AUC (sorted high → low):')
for _, row in (
    metrics_df[metrics_df['Label'] != 'MACRO AVG']
    .sort_values('AUC', ascending=False)
    .iterrows()
):
    bar = '█' * int(round(float(row['AUC']) * 30))
    print(f'  {row["Label"]:<22s} {row["AUC"]:.4f}  {bar}')
print('=' * 60)